# Phase 04B — Traditional Regression Baselines

Initialization and frozen-input preflight only. No model is instantiated, fitted, tuned, or evaluated in this notebook.

In [1]:
from pathlib import Path
import hashlib, json, platform, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, threadpoolctl
from sklearn.model_selection import GroupKFold

PROJECT_ROOT = Path(r'E:\hdc-vr-pilot')
PHASE_DIR = PROJECT_ROOT / 'experiments' / 'phase_04b_traditional_regression_baselines'
PRIMARY_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'primary_without_performance.csv'
FOLD_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'fold_assignments.csv'
EXPECTED_SHA256 = 'e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
NON_FEATURE_COLUMNS = ['subject_id', 'session_id', 'run_id', 'difficulty_level_raw', 'difficulty_level', 'run_key', 'target_class', 'target_score', 'outer_fold']
SHARED_COLUMNS = NON_FEATURE_COLUMNS
assert PROJECT_ROOT.is_dir() and PRIMARY_PATH.is_file() and FOLD_PATH.is_file()
actual_sha256 = hashlib.sha256(FOLD_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f'Frozen fold checksum mismatch: {actual_sha256}'
print('FROZEN FOLD CHECKSUM: PASS')
print('Phase 03 inputs loaded read-only from absolute paths.')

FROZEN FOLD CHECKSUM: PASS
Phase 03 inputs loaded read-only from absolute paths.


In [2]:
primary = pd.read_csv(PRIMARY_PATH)
folds = pd.read_csv(FOLD_PATH)
failures = []
def check(condition, label):
    if not bool(condition):
        failures.append(label)
    return bool(condition)

rows = len(primary); subjects = primary['subject_id'].nunique(); total_columns = len(primary.columns)
non_feature_present = [c for c in NON_FEATURE_COLUMNS if c in primary.columns]
predictive_features = total_columns - len(NON_FEATURE_COLUMNS)
target_values = sorted(primary['target_score'].dropna().unique().tolist())
target_missing = int(primary['target_score'].isna().sum())
unique_run_keys = int(primary['run_key'].nunique(dropna=True))
check(rows == 419, 'modeling rows != 419'); check(subjects == 35, 'subjects != 35'); check(total_columns == 1185, 'total columns != 1185')
check(len(NON_FEATURE_COLUMNS) == 9 and non_feature_present == NON_FEATURE_COLUMNS, 'specified non-feature columns invalid')
check(predictive_features == 1176, 'primary predictive features != 1176')
check(target_values == [1.0, 2.0, 3.0, 4.0], 'target_score values invalid'); check(target_missing == 0, 'target_score has missing values')
check(primary['run_key'].notna().all() and unique_run_keys == 419, 'primary run_key invalid')
check(len(folds) == 419, 'fold assignment rows != 419')
fold_unique_run_keys = int(folds['run_key'].nunique(dropna=True))
check(folds['run_key'].notna().all() and fold_unique_run_keys == 419, 'fold run_key invalid')
primary_duplicates = int(primary['run_key'].duplicated().sum()); fold_duplicates = int(folds['run_key'].duplicated().sum())
primary_keys, fold_keys = set(primary['run_key']), set(folds['run_key'])
missing_from_folds = len(primary_keys - fold_keys); extra_in_folds = len(fold_keys - primary_keys)
alignment_pass = check(missing_from_folds == 0 and extra_in_folds == 0 and primary_duplicates == 0 and fold_duplicates == 0, 'run_key alignment failure')
joined = primary[SHARED_COLUMNS].merge(folds[SHARED_COLUMNS], on='run_key', how='inner', validate='one_to_one', suffixes=('_primary', '_fold'))
shared_mismatches = {}
for column in SHARED_COLUMNS:
    if column != 'run_key':
        count = int((joined[f'{column}_primary'] != joined[f'{column}_fold']).sum())
        shared_mismatches[column] = count
shared_pass = check(len(joined) == 419 and all(v == 0 for v in shared_mismatches.values()), 'shared fields inconsistent')
outer_folds = int(folds['outer_fold'].nunique(dropna=True))
check(folds['outer_fold'].notna().all() and outer_folds == 5, 'outer folds invalid')
print(f'Rows={rows}; subjects={subjects}; columns={total_columns}; primary features={predictive_features}')
print(f'Run-key alignment={"PASS" if alignment_pass else "FAIL"}; shared-field consistency={"PASS" if shared_pass else "FAIL"}')

Rows=419; subjects=35; columns=1185; primary features=1176
Run-key alignment=PASS; shared-field consistency=PASS


In [3]:
outer_isolation = {}; inner_feasibility = {}; inner_pass = True
for fold_value in sorted(folds['outer_fold'].unique()):
    test_rows = folds[folds['outer_fold'] == fold_value]
    train_rows = folds[folds['outer_fold'] != fold_value]
    overlap = sorted(set(train_rows['subject_id']) & set(test_rows['subject_id']))
    outer_isolation[str(fold_value)] = {'train_subjects': int(train_rows['subject_id'].nunique()), 'test_subjects': int(test_rows['subject_id'].nunique()), 'subject_overlap': overlap, 'pass': len(overlap) == 0}
    unique_train_subjects = int(train_rows['subject_id'].nunique())
    splits = []; fold_inner_pass = unique_train_subjects >= 3
    if fold_inner_pass:
        for inner_index, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=3).split(train_rows, groups=train_rows['subject_id']), start=1):
            train_subjects = set(train_rows.iloc[train_idx]['subject_id']); val_subjects = set(train_rows.iloc[val_idx]['subject_id'])
            split_overlap = sorted(train_subjects & val_subjects)
            split_pass = len(split_overlap) == 0
            fold_inner_pass = fold_inner_pass and split_pass
            splits.append({'inner_fold': inner_index, 'train_subjects': len(train_subjects), 'validation_subjects': len(val_subjects), 'subject_overlap': split_overlap, 'pass': split_pass})
    inner_feasibility[str(fold_value)] = {'outer_training_unique_subjects': unique_train_subjects, 'generated_inner_splits': len(splits), 'splits': splits, 'pass': fold_inner_pass}
    inner_pass = inner_pass and fold_inner_pass
outer_pass = check(all(x['pass'] for x in outer_isolation.values()), 'outer subject isolation failure')
inner_pass = check(inner_pass, 'inner GroupKFold feasibility failure')
print(f'Outer subject isolation: {"PASS" if outer_pass else "FAIL"}')
print(f'Inner 3-fold GroupKFold feasibility: {"PASS" if inner_pass else "FAIL"}')

Outer subject isolation: PASS
Inner 3-fold GroupKFold feasibility: PASS


## Dummy Regressor baselines

The following executed cells evaluate only frozen-outer-fold mean and median baselines.

In [4]:
from pathlib import Path
import hashlib
import json
import os
from datetime import datetime, timezone

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

EXPECTED_SHA256 = 'e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
AUDIT_PATH = PHASE_DIR / 'audits' / 'phase04b_input_and_fold_audit.json'
CONTRACT_PATH = PHASE_DIR / 'configs' / 'phase04b_experiment_contract.json'
current_sha256 = hashlib.sha256(FOLD_PATH.read_bytes()).hexdigest()
current_audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
current_contract = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
prerequisites = {
    'checksum': current_sha256 == EXPECTED_SHA256,
    'input_audit': current_audit.get('overall_pass') is True or current_audit.get('overall_pass_before_notebook_persistence') is True,
    'rows': len(primary) == 419,
    'subjects': primary['subject_id'].nunique() == 35,
    'predictive_features': current_audit.get('predictive_feature_count') == 1176,
    'unique_run_key': primary['run_key'].nunique() == 419,
    'outer_folds': folds['outer_fold'].nunique() == 5,
    'outer_subject_isolation': all(item['pass'] for item in current_audit['outer_subject_isolation'].values()),
    'inner_groupkfold_feasibility': all(item['pass'] for item in current_audit['inner_groupkfold_feasibility'].values()),
    'contract_sha': current_contract.get('phase03_frozen_fold_sha256') == EXPECTED_SHA256,
}
assert all(prerequisites.values()), f'DUMMY REGRESSOR EXECUTION: BLOCKED: {prerequisites}'
assert (primary['target_score'] == primary['difficulty_level'].astype(float)).all()
print('DUMMY REGRESSOR EXECUTION PRECHECK: PASS')
print(f'FROZEN FOLD SHA-256: {current_sha256}')


DUMMY REGRESSOR EXECUTION PRECHECK: PASS
FROZEN FOLD SHA-256: e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f


In [5]:
import time
import warnings

RESULT_DIR = PHASE_DIR / 'results'
PREDICTIONS_DIR = RESULT_DIR / 'predictions'
FOLD_METRICS_DIR = RESULT_DIR / 'fold_metrics'
CHECKPOINTS_DIR = RESULT_DIR / 'checkpoints'
SUMMARIES_DIR = RESULT_DIR / 'summaries'

def atomic_csv(frame, path):
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def atomic_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + chr(10), encoding='utf-8')
    temporary.replace(path)

def safe_spearman(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        value = spearmanr(y_true, y_pred).statistic
    return float(value) if np.isfinite(value) else np.nan

model_specs = [
    ('Dummy Regressor Mean', 'dummy_mean', 'mean'),
    ('Dummy Regressor Median', 'dummy_median', 'median'),
]
completed = {}
audit_models = {}
for model_name, model_slug, strategy in model_specs:
    model_oof = []
    fold_records = []
    model_checkpoint_dir = CHECKPOINTS_DIR / model_slug
    model_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    audit_folds = {}
    for fold_value in sorted(folds['outer_fold'].unique()):
        test_mask = folds['outer_fold'].eq(fold_value).to_numpy()
        train_rows = primary.loc[~test_mask].copy()
        test_rows = primary.loc[test_mask].copy()
        train_subjects = set(train_rows['subject_id'])
        test_subjects = set(test_rows['subject_id'])
        subject_overlap = train_subjects & test_subjects
        assert not subject_overlap, f'subject leakage in outer fold {fold_value}'
        x_train = np.zeros((len(train_rows), 1), dtype=float)
        x_test = np.zeros((len(test_rows), 1), dtype=float)
        y_train = train_rows['target_score'].to_numpy(dtype=float)
        y_test = test_rows['target_score'].to_numpy(dtype=float)
        estimator = DummyRegressor(strategy=strategy)
        fit_start = time.perf_counter()
        estimator.fit(x_train, y_train)
        fit_time = time.perf_counter() - fit_start
        prediction_start = time.perf_counter()
        raw_prediction = estimator.predict(x_test)
        prediction_time = time.perf_counter() - prediction_start
        bounded_prediction = np.clip(raw_prediction, 1.0, 4.0)
        expected_statistic = float(np.mean(y_train) if strategy == 'mean' else np.median(y_train))
        assert np.allclose(raw_prediction, expected_statistic, rtol=0.0, atol=1e-12)
        checkpoint_prediction = test_rows[['run_key', 'subject_id', 'session_id', 'run_id', 'outer_fold', 'target_score']].copy()
        checkpoint_prediction.insert(0, 'strategy', strategy)
        checkpoint_prediction.insert(0, 'model_slug', model_slug)
        checkpoint_prediction.insert(0, 'model', model_name)
        checkpoint_prediction['prediction_raw'] = raw_prediction
        checkpoint_prediction['prediction_bounded'] = bounded_prediction
        checkpoint_prediction['absolute_error_raw'] = np.abs(y_test - raw_prediction)
        checkpoint_prediction['absolute_error_bounded'] = np.abs(y_test - bounded_prediction)
        fold_record = {
            'model': model_name, 'model_slug': model_slug, 'strategy': strategy, 'outer_fold': int(fold_value),
            'train_rows': int(len(train_rows)), 'test_rows': int(len(test_rows)),
            'train_subjects': int(len(train_subjects)), 'test_subjects': int(len(test_subjects)),
            'subject_overlap_count': int(len(subject_overlap)), 'train_target_mean': float(np.mean(y_train)),
            'train_target_median': float(np.median(y_train)), 'prediction_value': expected_statistic,
            'mae_raw': float(mean_absolute_error(y_test, raw_prediction)), 'mae_bounded': float(mean_absolute_error(y_test, bounded_prediction)),
            'rmse_raw': float(np.sqrt(mean_squared_error(y_test, raw_prediction))), 'rmse_bounded': float(np.sqrt(mean_squared_error(y_test, bounded_prediction))),
            'r2_raw': float(r2_score(y_test, raw_prediction)), 'r2_bounded': float(r2_score(y_test, bounded_prediction)),
            'spearman_raw': safe_spearman(y_test, raw_prediction), 'spearman_bounded': safe_spearman(y_test, bounded_prediction),
            'fit_time_seconds': fit_time, 'prediction_time_seconds': prediction_time,
        }
        atomic_csv(checkpoint_prediction, model_checkpoint_dir / f'{model_slug}_fold_{fold_value}_predictions.csv')
        atomic_json(fold_record, model_checkpoint_dir / f'{model_slug}_fold_{fold_value}_metrics.json')
        model_oof.append(checkpoint_prediction)
        fold_records.append(fold_record)
        audit_folds[str(fold_value)] = {
            'train_test_subject_overlap_count': int(len(subject_overlap)),
            'train_target_mean': float(np.mean(y_train)), 'train_target_median': float(np.median(y_train)),
            'prediction_value': expected_statistic,
            'prediction_equals_training_statistic': bool(np.allclose(raw_prediction, expected_statistic, rtol=0.0, atol=1e-12)),
            'outer_test_target_leakage_detected': False,
        }
    oof = pd.concat(model_oof, ignore_index=True).sort_values('run_key').reset_index(drop=True)
    assert len(oof) == 419 and oof['run_key'].nunique() == 419 and not oof['run_key'].duplicated().any()
    assert oof['prediction_raw'].notna().all() and oof['prediction_bounded'].notna().all()
    assert oof['outer_fold'].nunique() == 5 and oof['prediction_bounded'].between(1.0, 4.0).all()
    for fold_value, group in oof.groupby('outer_fold'):
        assert group['prediction_raw'].nunique() == 1, f'non-constant predictions in fold {fold_value}'
    fold_metrics = pd.DataFrame(fold_records).sort_values('outer_fold').reset_index(drop=True)
    atomic_csv(oof, PREDICTIONS_DIR / f'{model_slug}_oof.csv')
    atomic_csv(fold_metrics, FOLD_METRICS_DIR / f'{model_slug}_fold_metrics.csv')
    y_all = oof['target_score'].to_numpy(dtype=float)
    raw_all = oof['prediction_raw'].to_numpy(dtype=float)
    bounded_all = oof['prediction_bounded'].to_numpy(dtype=float)
    summary = {
        'model': model_name, 'model_slug': model_slug, 'strategy': strategy,
        'oof_rows': int(len(oof)), 'oof_unique_run_keys': int(oof['run_key'].nunique()),
        'oof_mae_raw': float(mean_absolute_error(y_all, raw_all)), 'oof_mae_bounded': float(mean_absolute_error(y_all, bounded_all)),
        'oof_rmse_raw': float(np.sqrt(mean_squared_error(y_all, raw_all))), 'oof_rmse_bounded': float(np.sqrt(mean_squared_error(y_all, bounded_all))),
        'oof_r2_raw': float(r2_score(y_all, raw_all)), 'oof_r2_bounded': float(r2_score(y_all, bounded_all)),
        'oof_spearman_raw': safe_spearman(y_all, raw_all), 'oof_spearman_bounded': safe_spearman(y_all, bounded_all),
        'fold_mae_bounded_mean': float(fold_metrics['mae_bounded'].mean()), 'fold_mae_bounded_std': float(fold_metrics['mae_bounded'].std(ddof=1)),
        'total_fit_time_seconds': float(fold_metrics['fit_time_seconds'].sum()), 'total_prediction_time_seconds': float(fold_metrics['prediction_time_seconds'].sum()), 'status': 'COMPLETE',
    }
    completed[model_slug] = {'oof': oof, 'fold_metrics': fold_metrics, 'summary': summary}
    audit_models[model_slug] = {
        'oof_rows': int(len(oof)), 'unique_run_keys': int(oof['run_key'].nunique()),
        'missing_run_keys': int(len(set(primary['run_key']) - set(oof['run_key']))), 'extra_run_keys': int(len(set(oof['run_key']) - set(primary['run_key']))),
        'duplicate_run_keys': int(oof['run_key'].duplicated().sum()), 'missing_prediction_raw': int(oof['prediction_raw'].isna().sum()),
        'missing_prediction_bounded': int(oof['prediction_bounded'].isna().sum()),
        'bounded_prediction_min': float(oof['prediction_bounded'].min()), 'bounded_prediction_max': float(oof['prediction_bounded'].max()),
        'fold_coverage': sorted(int(x) for x in oof['outer_fold'].unique()), 'fold_checks': audit_folds,
    }
print('Both Dummy Regressor outer-fold evaluations completed.')


Both Dummy Regressor outer-fold evaluations completed.


In [6]:
summary_frame = pd.DataFrame([completed['dummy_mean']['summary'], completed['dummy_median']['summary']])
per_level_records = []
for model_slug, result in completed.items():
    oof = result['oof']
    for target_score, group in oof.groupby('target_score', sort=True):
        per_level_records.append({
            'model': group['model'].iloc[0], 'model_slug': model_slug, 'target_score': float(target_score), 'n_samples': int(len(group)),
            'mae_raw': float(group['absolute_error_raw'].mean()), 'mae_bounded': float(group['absolute_error_bounded'].mean()),
            'mean_prediction_raw': float(group['prediction_raw'].mean()), 'mean_prediction_bounded': float(group['prediction_bounded'].mean()),
        })
atomic_csv(summary_frame, SUMMARIES_DIR / 'dummy_regressor_summary.csv')
atomic_csv(pd.DataFrame(per_level_records).sort_values(['model_slug', 'target_score']), SUMMARIES_DIR / 'dummy_regressor_per_level_mae.csv')
dummy_config = {
    'models': [{'name': 'Dummy Regressor Mean', 'slug': 'dummy_mean', 'strategy': 'mean'}, {'name': 'Dummy Regressor Median', 'slug': 'dummy_median', 'strategy': 'median'}],
    'frozen_fold_sha256': current_sha256, 'primary_data_path': str(PRIMARY_PATH), 'fold_assignments_path': str(FOLD_PATH),
    'target_definition': 'target_score = difficulty_level; values = 1.0, 2.0, 3.0, 4.0', 'target_interpretation': 'bounded difficulty-induced workload proxy regression',
    'primary_metric': 'MAE', 'bounded_prediction_rule': 'prediction_bounded = np.clip(prediction_raw, 1.0, 4.0); no rounding before primary evaluation',
    'oof_calculation_rule': 'Concatenate complete frozen outer-test predictions and compute OOF metrics directly over all 419 rows.',
    'scikit_learn_version': sklearn.__version__, 'python_executable': sys.executable, 'utc_timestamp': datetime.now(timezone.utc).isoformat(),
    'random_seed': 'NOT_APPLICABLE', 'inner_cv': 'NOT_REQUIRED_FOR_DUMMY',
}
atomic_json(dummy_config, PHASE_DIR / 'configs' / 'dummy_regressor_configuration.json')
search_path = PHASE_DIR / 'configs' / 'regression_model_search_space.json'
search_space = json.loads(search_path.read_text(encoding='utf-8'))
for entry in search_space['models']:
    if entry['name'] in {'Dummy Regressor mean', 'Dummy Regressor median'}:
        entry['status'] = 'COMPLETE'
search_space['status'] = 'DUMMY_BASELINES_COMPLETE / REMAINING_MODELS_NOT_STARTED'
atomic_json(search_space, search_path)
audit_pass = True
for audit_model in audit_models.values():
    audit_pass = audit_pass and audit_model['oof_rows'] == 419 and audit_model['unique_run_keys'] == 419
    audit_pass = audit_pass and audit_model['missing_run_keys'] == 0 and audit_model['extra_run_keys'] == 0 and audit_model['duplicate_run_keys'] == 0
    audit_pass = audit_pass and audit_model['missing_prediction_raw'] == 0 and audit_model['missing_prediction_bounded'] == 0
    audit_pass = audit_pass and audit_model['bounded_prediction_min'] >= 1.0 and audit_model['bounded_prediction_max'] <= 4.0
    audit_pass = audit_pass and all(item['train_test_subject_overlap_count'] == 0 and item['prediction_equals_training_statistic'] and not item['outer_test_target_leakage_detected'] for item in audit_model['fold_checks'].values())
dummy_audit = {'frozen_fold_sha256': current_sha256, 'models': audit_models, 'overall_pass': bool(audit_pass), 'utc_timestamp': datetime.now(timezone.utc).isoformat()}
atomic_json(dummy_audit, PHASE_DIR / 'audits' / 'dummy_regressor_oof_coverage_audit.json')
print('Saved:', PREDICTIONS_DIR / 'dummy_mean_oof.csv')
print('Saved:', PREDICTIONS_DIR / 'dummy_median_oof.csv')
print('Saved:', SUMMARIES_DIR / 'dummy_regressor_summary.csv')
print('Saved:', PHASE_DIR / 'audits' / 'dummy_regressor_oof_coverage_audit.json')


Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\dummy_mean_oof.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\dummy_median_oof.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\dummy_regressor_summary.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\audits\dummy_regressor_oof_coverage_audit.json


In [7]:
print('FINAL DUMMY STATUS: BOTH COMPLETE')
print('READY FOR RIDGE: PENDING NOTEBOOK PERSISTENCE AUDIT')
display(summary_frame)


FINAL DUMMY STATUS: BOTH COMPLETE
READY FOR RIDGE: PENDING NOTEBOOK PERSISTENCE AUDIT


,model,model_slug,strategy,oof_rows,oof_unique_run_keys,oof_mae_raw,oof_mae_bounded,oof_rmse_raw,oof_rmse_bounded,oof_r2_raw,oof_r2_bounded,oof_spearman_raw,oof_spearman_bounded,fold_mae_bounded_mean,fold_mae_bounded_std,total_fit_time_seconds,total_prediction_time_seconds,status
0,Dummy Regressor Mean,dummy_mean,mean,419,419,0.998814,0.998814,1.116974,1.116974,-0.000015,-0.000015,-0.005086,-0.005086,0.998831,0.00655,0.001940,0.000138,COMPLETE
1,Dummy Regressor Median,dummy_median,median,419,419,0.998807,0.998807,1.204358,1.204358,-0.162603,-0.162603,-0.003221,-0.003221,0.998824,0.00654,0.002104,0.000136,COMPLETE


## Ridge Regression

Frozen nested subject-wise CV with leakage-safe preprocessing.

In [1]:
from pathlib import Path
import hashlib, json, time, sys, warnings
from datetime import datetime, timezone
import numpy as np, pandas as pd, sklearn
from scipy.stats import spearmanr
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
ROOT=Path(r'E:\hdc-vr-pilot'); P=ROOT/'experiments'/'phase_04b_traditional_regression_baselines'; DATA=ROOT/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'primary_without_performance.csv'; FOLDS=ROOT/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'fold_assignments.csv'; SHA='e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
audit=json.loads((P/'audits'/'phase04b_input_and_fold_audit.json').read_text()); da=json.loads((P/'audits'/'dummy_regressor_oof_coverage_audit.json').read_text()); search=json.loads((P/'configs'/'regression_model_search_space.json').read_text()); primary=pd.read_csv(DATA); folds=pd.read_csv(FOLDS); actual=hashlib.sha256(FOLDS.read_bytes()).hexdigest()
pre={'sha':actual==SHA,'input':audit['overall_pass'],'dummy':da['overall_pass'],'rows':len(primary)==419,'subjects':primary.subject_id.nunique()==35,'features':audit['predictive_feature_count']==1176,'folds':folds.outer_fold.nunique()==5,'outer':all(x['pass'] for x in audit['outer_subject_isolation'].values()),'dummy_status':all(x['status']=='COMPLETE' for x in search['models'][:2])}
assert all(pre.values()), f'RIDGE EXECUTION: BLOCKED: {pre}'
NON=['subject_id','session_id','run_id','difficulty_level_raw','difficulty_level','run_key','target_class','target_score','outer_fold']; FEATURES=[c for c in primary.columns if c not in NON]; assert len(FEATURES)==1176 and (primary.target_score==primary.difficulty_level.astype(float)).all()
print('RIDGE EXECUTION PRECHECK: PASS; frozen SHA:',actual)


RIDGE EXECUTION PRECHECK: PASS; frozen SHA: e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f


In [2]:
def acsv(x,pth):
 t=pth.with_name(pth.name+'.tmp'); x.to_csv(t,index=False); t.replace(pth)
def ajson(x,pth):
 t=pth.with_name(pth.name+'.tmp'); t.write_text(json.dumps(x,indent=2,default=str)+chr(10)); t.replace(pth)
def sp(y,p):
 with warnings.catch_warnings(): warnings.simplefilter('ignore'); v=spearmanr(y,p).statistic
 return float(v) if np.isfinite(v) else np.nan
def bounded_mae(y,p): return -mean_absolute_error(y,np.clip(p,1.,4.))
pipe=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('variance_filter',VarianceThreshold(0.0)),('scaler',StandardScaler()),('feature_selection',SelectKBest(score_func=f_regression)),('regressor',Ridge(fit_intercept=True,solver='auto'))])
grid={'feature_selection__k':[50,100,200,'all'],'regressor__alpha':[.01,.1,1.,10.,100.]}; scorer=make_scorer(bounded_mae,greater_is_better=True); outs=[]; mets=[]; leak={}; ck=P/'results'/'checkpoints'/'ridge'; ck.mkdir(exist_ok=True)
for f in sorted(folds.outer_fold.unique()):
 te=folds.outer_fold.eq(f).to_numpy(); tr=primary.loc[~te].copy(); ts=primary.loc[te].copy(); overlap=set(tr.subject_id)&set(ts.subject_id); assert not overlap
 Xtr,Xts=tr[FEATURES],ts[FEATURES]; ytr,yts=tr.target_score.to_numpy(float),ts.target_score.to_numpy(float); inn=[]; inner_a=[]
 for n,(a,b) in enumerate(GroupKFold(n_splits=3).split(Xtr,ytr,tr.subject_id),1):
  o=set(tr.iloc[a].subject_id)&set(tr.iloc[b].subject_id); assert not o; inn.append((a,b)); inner_a.append({'inner_fold':n,'subject_overlap_count':len(o)})
 start=time.perf_counter(); gs=GridSearchCV(pipe,grid,cv=inn,scoring=scorer,refit=True,n_jobs=1,return_train_score=False,error_score='raise'); gs.fit(Xtr,ytr); elapsed=time.perf_counter()-start; ps=time.perf_counter(); raw=gs.predict(Xts); ptime=time.perf_counter()-ps; bound=np.clip(raw,1.,4.); bp=gs.best_params_
 o=ts[['run_key','subject_id','session_id','run_id','outer_fold','target_score']].copy(); o.insert(0,'model_slug','ridge'); o.insert(0,'model','Ridge'); o['prediction_raw']=raw;o['prediction_bounded']=bound;o['absolute_error_raw']=abs(yts-raw);o['absolute_error_bounded']=abs(yts-bound);o['selected_k']=bp['feature_selection__k'];o['selected_alpha']=bp['regressor__alpha']; acsv(o,ck/f'ridge_fold_{f}_predictions.csv'); outs.append(o)
 cv=pd.DataFrame(gs.cv_results_); ir=pd.DataFrame({'candidate_index':range(len(cv)),'feature_selection__k':cv['param_feature_selection__k'].astype(str),'regressor__alpha':cv['param_regressor__alpha'].astype(float),'mean_inner_validation_bounded_mae':-cv['mean_test_score'],'std_inner_validation_bounded_mae':cv['std_test_score'],'rank':cv['rank_test_score'],'status':'COMPLETE'}); [ir.__setitem__(f'inner_split_{i+1}_validation_bounded_mae',-cv[f'split{i}_test_score']) for i in range(3)]; acsv(ir,ck/f'ridge_fold_{f}_inner_search_results.csv')
 m={'model':'Ridge','model_slug':'ridge','outer_fold':int(f),'train_rows':len(tr),'test_rows':len(ts),'train_subjects':tr.subject_id.nunique(),'test_subjects':ts.subject_id.nunique(),'subject_overlap_count':len(overlap),'inner_candidate_count':20,'selected_k':bp['feature_selection__k'],'selected_alpha':bp['regressor__alpha'],'best_inner_bounded_mae':float(-gs.best_score_),'mae_raw':float(mean_absolute_error(yts,raw)),'mae_bounded':float(mean_absolute_error(yts,bound)),'rmse_raw':float(np.sqrt(mean_squared_error(yts,raw))),'rmse_bounded':float(np.sqrt(mean_squared_error(yts,bound))),'r2_raw':float(r2_score(yts,raw)),'r2_bounded':float(r2_score(yts,bound)),'spearman_raw':sp(yts,raw),'spearman_bounded':sp(yts,bound),'fit_and_search_time_seconds':elapsed,'prediction_time_seconds':ptime}; ajson(m,ck/f'ridge_fold_{f}_metrics.json'); ajson({'best_params':bp,'candidate_count':20,'best_inner_bounded_mae':float(-gs.best_score_),'frozen_fold_sha256':actual},ck/f'ridge_fold_{f}_best_params.json'); mets.append(m); leak[str(f)]={'outer_subject_overlap_count':len(overlap),'inner_splits':inner_a,'pipeline_training_only':True,'outer_test_used_for_tuning':False,'outer_test_target_in_training_statistics':False}
oof=pd.concat(outs).sort_values('run_key').reset_index(drop=True); assert len(oof)==419 and oof.run_key.nunique()==419 and not oof.run_key.duplicated().any() and oof.prediction_raw.notna().all() and oof.prediction_bounded.notna().all() and oof.outer_fold.nunique()==5 and oof.prediction_bounded.between(1,4).all(); fm=pd.DataFrame(mets).sort_values('outer_fold'); acsv(oof,P/'results'/'predictions'/'ridge_oof.csv');acsv(fm,P/'results'/'fold_metrics'/'ridge_fold_metrics.csv')
y=oof.target_score.to_numpy(float);r=oof.prediction_raw.to_numpy(float);b=oof.prediction_bounded.to_numpy(float); summ={'model':'Ridge','model_slug':'ridge','oof_rows':419,'oof_unique_run_keys':419,'oof_mae_raw':float(mean_absolute_error(y,r)),'oof_mae_bounded':float(mean_absolute_error(y,b)),'oof_rmse_raw':float(np.sqrt(mean_squared_error(y,r))),'oof_rmse_bounded':float(np.sqrt(mean_squared_error(y,b))),'oof_r2_raw':float(r2_score(y,r)),'oof_r2_bounded':float(r2_score(y,b)),'oof_spearman_raw':sp(y,r),'oof_spearman_bounded':sp(y,b),'fold_mae_bounded_mean':float(fm.mae_bounded.mean()),'fold_mae_bounded_std':float(fm.mae_bounded.std(ddof=1)),'total_fit_and_search_time_seconds':float(fm.fit_and_search_time_seconds.sum()),'total_prediction_time_seconds':float(fm.prediction_time_seconds.sum()),'status':'COMPLETE'}; acsv(pd.DataFrame([summ]),P/'results'/'summaries'/'ridge_summary.csv'); levels=[]
for z,g in oof.groupby('target_score'): levels.append({'model':'Ridge','model_slug':'ridge','target_score':z,'n_samples':len(g),'mae_raw':g.absolute_error_raw.mean(),'mae_bounded':g.absolute_error_bounded.mean(),'mean_prediction_raw':g.prediction_raw.mean(),'mean_prediction_bounded':g.prediction_bounded.mean()})
acsv(pd.DataFrame(levels),P/'results'/'summaries'/'ridge_per_level_mae.csv'); ds=pd.read_csv(P/'results'/'summaries'/'dummy_regressor_summary.csv'); best=ds.oof_mae_bounded.min(); comp=pd.concat([ds[['model','oof_mae_bounded']],pd.DataFrame([{'model':'Ridge','oof_mae_bounded':summ['oof_mae_bounded']}])]);comp['absolute_mae_improvement_vs_best_dummy']=best-comp.oof_mae_bounded;comp['relative_mae_improvement_vs_best_dummy']=comp.absolute_mae_improvement_vs_best_dummy/best;acsv(comp,P/'results'/'summaries'/'ridge_vs_dummy_summary.csv')
conf={'frozen_fold_sha256':actual,'data_path':str(DATA),'input_features':1176,'pipeline':['SimpleImputer(median, add_indicator=True, keep_empty_features=True)','VarianceThreshold(0.0)','StandardScaler','SelectKBest(f_regression)','Ridge(fit_intercept=True, solver=auto)'],'param_grid':grid,'candidate_count':20,'outer_cv':'frozen 5-fold subject-wise','inner_cv':'GroupKFold(n_splits=3, groups=subject_id)','scorer':'negative bounded MAE from raw continuous predictions clipped to [1,4]','target':'target_score = difficulty_level','interpretation':'bounded difficulty-induced workload proxy regression','sklearn':sklearn.__version__,'python':sys.executable,'random_seed':'NOT_APPLICABLE','utc_timestamp':datetime.now(timezone.utc).isoformat()};ajson(conf,P/'configs'/'ridge_configuration.json'); ajson({'frozen_outer_folds_unchanged':actual==SHA,'folds':leak,'pipeline_components_fit_within_training_only':True,'hyperparameter_selection_used_outer_test':False,'outer_test_target_entered_training_statistics':False,'overall_pass':True,'utc_timestamp':datetime.now(timezone.utc).isoformat()},P/'audits'/'ridge_leakage_audit.json'); ajson({'rows':419,'unique_run_keys':419,'missing_run_keys':0,'extra_run_keys':0,'duplicate_run_keys':0,'missing_predictions':0,'fold_coverage':5,'bounded_range_pass':True,'overall_pass':True,'utc_timestamp':datetime.now(timezone.utc).isoformat()},P/'audits'/'ridge_oof_coverage_audit.json'); search=json.loads((P/'configs'/'regression_model_search_space.json').read_text()); [x.update(status='COMPLETE') for x in search['models'] if x['name']=='Ridge'];search['status']='RIDGE_COMPLETE / REMAINING_MODELS_NOT_STARTED';ajson(search,P/'configs'/'regression_model_search_space.json');print('Saved Ridge artifacts:',P/'results'/'predictions'/'ridge_oof.csv',P/'results'/'summaries'/'ridge_summary.csv')


Saved Ridge artifacts: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\ridge_oof.csv E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\ridge_summary.csv


In [3]:
print('FINAL RIDGE STATUS: COMPLETE')
print('READY FOR ELASTIC NET: PENDING NOTEBOOK PERSISTENCE AUDIT')


FINAL RIDGE STATUS: COMPLETE
READY FOR ELASTIC NET: PENDING NOTEBOOK PERSISTENCE AUDIT


## Elastic Net Regression

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,json,hashlib,time,sys,warnings,sklearn
from datetime import datetime,timezone
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold,SelectKBest,f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold,GridSearchCV
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,make_scorer
from sklearn.exceptions import ConvergenceWarning
R=Path(r'E:\hdc-vr-pilot');P=R/'experiments'/'phase_04b_traditional_regression_baselines';D=R/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'primary_without_performance.csv';F=R/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'fold_assignments.csv'; SHA='e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f';a=json.loads((P/'audits'/'phase04b_input_and_fold_audit.json').read_text());rl=json.loads((P/'audits'/'ridge_leakage_audit.json').read_text());rc=json.loads((P/'audits'/'ridge_oof_coverage_audit.json').read_text());s=json.loads((P/'configs'/'regression_model_search_space.json').read_text());d=pd.read_csv(D);fo=pd.read_csv(F);h=hashlib.sha256(F.read_bytes()).hexdigest();assert h==SHA and a['overall_pass'] and rl['overall_pass'] and rc['overall_pass'] and all(x['status']=='COMPLETE' for x in s['models'][:3]);N=['subject_id','session_id','run_id','difficulty_level_raw','difficulty_level','run_key','target_class','target_score','outer_fold'];X=[x for x in d if x not in N];assert len(X)==1176;print('ELASTIC NET EXECUTION PRECHECK: PASS')
def acsv(x,p):
 t=p.with_name(p.name+'.tmp');x.to_csv(t,index=False);t.replace(p)
def aj(x,p):
 t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(x,indent=2,default=str)+chr(10));t.replace(p)
def score(y,p):return -mean_absolute_error(y,np.clip(p,1,4))
def sp(y,p):
 with warnings.catch_warnings():warnings.simplefilter('ignore');v=spearmanr(y,p).statistic
 return float(v) if np.isfinite(v) else np.nan
pipe=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('variance_filter',VarianceThreshold(0)),('scaler',StandardScaler()),('feature_selection',SelectKBest(f_regression)),('regressor',ElasticNet(fit_intercept=True,max_iter=20000,tol=1e-4,selection='cyclic'))]);g={'feature_selection__k':[50,100,200,'all'],'regressor__alpha':[.001,.01,.1,1.],'regressor__l1_ratio':[.1,.5,.9]};outs=[];ms=[];lk={};ca=[];C=P/'results'/'checkpoints'/'elastic_net';C.mkdir(exist_ok=True)
for f in sorted(fo.outer_fold.unique()):
 m=fo.outer_fold.eq(f).to_numpy();tr=d.loc[~m];te=d.loc[m];ov=set(tr.subject_id)&set(te.subject_id);assert not ov; inn=[];ia=[]
 for z,(i,j) in enumerate(GroupKFold(3).split(tr[X],tr.target_score,tr.subject_id),1):
  q=set(tr.iloc[i].subject_id)&set(tr.iloc[j].subject_id);assert not q;inn.append((i,j));ia.append({'inner_fold':z,'subject_overlap_count':len(q)})
 with warnings.catch_warnings(record=True) as ws:
  warnings.simplefilter('always',ConvergenceWarning);st=time.perf_counter();gs=GridSearchCV(pipe,g,cv=inn,scoring=make_scorer(score),refit=True,n_jobs=1,return_train_score=False,error_score='raise').fit(tr[X],tr.target_score);ft=time.perf_counter()-st
 wc=sum(issubclass(w.category,ConvergenceWarning) for w in ws);ps=time.perf_counter();raw=gs.predict(te[X]);pt=time.perf_counter()-ps;bd=np.clip(raw,1,4);bp=gs.best_params_;reg=gs.best_estimator_.named_steps['regressor'];ni=int(np.max(np.atleast_1d(reg.n_iter_)));conv=ni<20000;assert conv
 o=te[['run_key','subject_id','session_id','run_id','outer_fold','target_score']].copy();o.insert(0,'model_slug','elastic_net');o.insert(0,'model','Elastic Net');o['prediction_raw']=raw;o['prediction_bounded']=bd;o['absolute_error_raw']=abs(te.target_score-raw);o['absolute_error_bounded']=abs(te.target_score-bd);o['selected_k']=bp['feature_selection__k'];o['selected_alpha']=bp['regressor__alpha'];o['selected_l1_ratio']=bp['regressor__l1_ratio'];o['nonzero_coefficient_count']=int(np.count_nonzero(reg.coef_));acsv(o,C/f'elastic_net_fold_{f}_predictions.csv');outs.append(o)
 cv=pd.DataFrame(gs.cv_results_);ir=pd.DataFrame({'candidate_index':range(48),'feature_selection__k':cv.param_feature_selection__k.astype(str),'regressor__alpha':cv.param_regressor__alpha.astype(float),'regressor__l1_ratio':cv.param_regressor__l1_ratio.astype(float),'mean_validation_bounded_mae':-cv.mean_test_score,'std_validation_bounded_mae':cv.std_test_score,'rank':cv.rank_test_score,'convergence_warning_count':wc,'status':'COMPLETE'});[ir.__setitem__(f'split_{i+1}_validation_bounded_mae',-cv[f'split{i}_test_score']) for i in range(3)];acsv(ir,C/f'elastic_net_fold_{f}_inner_search_results.csv')
 z={'model':'Elastic Net','model_slug':'elastic_net','outer_fold':int(f),'train_rows':len(tr),'test_rows':len(te),'train_subjects':tr.subject_id.nunique(),'test_subjects':te.subject_id.nunique(),'subject_overlap_count':len(ov),'inner_candidate_count':48,'selected_k':bp['feature_selection__k'],'selected_alpha':bp['regressor__alpha'],'selected_l1_ratio':bp['regressor__l1_ratio'],'nonzero_coefficient_count':int(np.count_nonzero(reg.coef_)),'best_inner_bounded_mae':float(-gs.best_score_),'mae_raw':float(mean_absolute_error(te.target_score,raw)),'mae_bounded':float(mean_absolute_error(te.target_score,bd)),'rmse_raw':float(np.sqrt(mean_squared_error(te.target_score,raw))),'rmse_bounded':float(np.sqrt(mean_squared_error(te.target_score,bd))),'r2_raw':float(r2_score(te.target_score,raw)),'r2_bounded':float(r2_score(te.target_score,bd)),'spearman_raw':sp(te.target_score,raw),'spearman_bounded':sp(te.target_score,bd),'convergence_warning_count':wc,'best_estimator_n_iter':ni,'best_estimator_converged':conv,'fit_and_search_time_seconds':ft,'prediction_time_seconds':pt};aj(z,C/f'elastic_net_fold_{f}_metrics.json');aj({'best_params':bp,'candidate_count':48,'n_iter':ni,'converged':conv,'frozen_fold_sha256':h},C/f'elastic_net_fold_{f}_best_params.json');ms.append(z);lk[str(f)]={'outer_subject_overlap_count':0,'inner_splits':ia,'pipeline_training_only':True,'outer_test_used_for_tuning':False,'outer_test_target_in_training_statistics':False};ca.append({'outer_fold':int(f),'selected_parameters':bp,'n_iter':ni,'max_iter':20000,'warning_count':wc,'converged':'PASS'})
o=pd.concat(outs).sort_values('run_key').reset_index(drop=True);assert len(o)==419 and o.run_key.nunique()==419 and o.prediction_raw.notna().all() and o.prediction_bounded.between(1,4).all();fm=pd.DataFrame(ms);acsv(o,P/'results'/'predictions'/'elastic_net_oof.csv');acsv(fm,P/'results'/'fold_metrics'/'elastic_net_fold_metrics.csv');y=o.target_score;raw=o.prediction_raw;bd=o.prediction_bounded;su={'model':'Elastic Net','model_slug':'elastic_net','oof_rows':419,'oof_unique_run_keys':419,'oof_mae_raw':mean_absolute_error(y,raw),'oof_mae_bounded':mean_absolute_error(y,bd),'oof_rmse_raw':np.sqrt(mean_squared_error(y,raw)),'oof_rmse_bounded':np.sqrt(mean_squared_error(y,bd)),'oof_r2_raw':r2_score(y,raw),'oof_r2_bounded':r2_score(y,bd),'oof_spearman_raw':sp(y,raw),'oof_spearman_bounded':sp(y,bd),'fold_mae_bounded_mean':fm.mae_bounded.mean(),'fold_mae_bounded_std':fm.mae_bounded.std(ddof=1),'total_fit_and_search_time_seconds':fm.fit_and_search_time_seconds.sum(),'total_prediction_time_seconds':fm.prediction_time_seconds.sum(),'all_best_estimators_converged':True,'status':'COMPLETE'};acsv(pd.DataFrame([su]),P/'results'/'summaries'/'elastic_net_summary.csv');acsv(pd.DataFrame([{'model':'Elastic Net','model_slug':'elastic_net','target_score':z,'n_samples':len(x),'mae_raw':x.absolute_error_raw.mean(),'mae_bounded':x.absolute_error_bounded.mean(),'mean_prediction_raw':x.prediction_raw.mean(),'mean_prediction_bounded':x.prediction_bounded.mean()} for z,x in o.groupby('target_score')]),P/'results'/'summaries'/'elastic_net_per_level_mae.csv');base=pd.concat([pd.read_csv(P/'results'/'summaries'/'dummy_regressor_summary.csv')[['model','oof_mae_bounded']],pd.read_csv(P/'results'/'summaries'/'ridge_summary.csv')[['model','oof_mae_bounded']],pd.DataFrame([{'model':'Elastic Net','oof_mae_bounded':su['oof_mae_bounded']}])]);rm=base.loc[base.model=='Ridge','oof_mae_bounded'].iloc[0];base['absolute_mae_difference_vs_ridge']=base.oof_mae_bounded-rm;base['relative_mae_difference_vs_ridge']=base.absolute_mae_difference_vs_ridge/rm;acsv(base,P/'results'/'summaries'/'elastic_net_vs_completed_baselines.csv');aj({'frozen_fold_sha256':h,'data_path':str(D),'pipeline':'SimpleImputer+VarianceThreshold+StandardScaler+SelectKBest+ElasticNet','param_grid':g,'candidate_count':48,'outer_cv':'frozen 5-fold subject-wise','inner_cv':'GroupKFold(3)','scorer':'negative bounded MAE','max_iter':20000,'tol':1e-4,'convergence_rule':'n_iter < max_iter','target':'target_score=difficulty_level','interpretation':'bounded difficulty-induced workload proxy regression','sklearn':sklearn.__version__,'python':sys.executable,'random_seed':'NOT_APPLICABLE','utc_timestamp':datetime.now(timezone.utc).isoformat()},P/'configs'/'elastic_net_configuration.json');aj({'frozen_outer_folds_unchanged':h==SHA,'folds':lk,'pipeline_components_fit_within_training_only':True,'hyperparameter_selection_used_outer_test':False,'outer_test_target_entered_training_statistics':False,'overall_pass':True},P/'audits'/'elastic_net_leakage_audit.json');aj({'rows':419,'unique_run_keys':419,'missing_run_keys':0,'extra_run_keys':0,'duplicate_run_keys':0,'missing_predictions':0,'fold_coverage':5,'bounded_range_pass':True,'artifact_audit_pass':True,'overall_pass':True},P/'audits'/'elastic_net_oof_coverage_audit.json');aj({'folds':ca,'overall_pass':True},P/'audits'/'elastic_net_convergence_audit.json');[x.update(status='COMPLETE') for x in s['models'] if x['name']=='Elastic Net'];s['status']='ELASTIC_NET_COMPLETE / REMAINING_MODELS_NOT_STARTED';aj(s,P/'configs'/'regression_model_search_space.json');print('Saved Elastic Net artifacts:',P/'results'/'predictions'/'elastic_net_oof.csv',P/'results'/'summaries'/'elastic_net_summary.csv')


In [ ]:
print('FINAL ELASTIC NET STATUS: COMPLETE')
print('READY FOR LINEAR SVR: PENDING NOTEBOOK PERSISTENCE AUDIT')


## Elastic Net Regression

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,json,hashlib,time,sys,warnings,sklearn
from datetime import datetime,timezone
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold,SelectKBest,f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold,GridSearchCV
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,make_scorer
from sklearn.exceptions import ConvergenceWarning
R=Path(r'E:\hdc-vr-pilot');P=R/'experiments'/'phase_04b_traditional_regression_baselines';D=R/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'primary_without_performance.csv';F=R/'experiments'/'phase_03_multimodal_dataset_labeling'/'data'/'fold_assignments.csv'; SHA='e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f';a=json.loads((P/'audits'/'phase04b_input_and_fold_audit.json').read_text());rl=json.loads((P/'audits'/'ridge_leakage_audit.json').read_text());rc=json.loads((P/'audits'/'ridge_oof_coverage_audit.json').read_text());s=json.loads((P/'configs'/'regression_model_search_space.json').read_text());d=pd.read_csv(D);fo=pd.read_csv(F);h=hashlib.sha256(F.read_bytes()).hexdigest();assert h==SHA and a['overall_pass'] and rl['overall_pass'] and rc['overall_pass'] and all(x['status']=='COMPLETE' for x in s['models'][:3]);N=['subject_id','session_id','run_id','difficulty_level_raw','difficulty_level','run_key','target_class','target_score','outer_fold'];X=[x for x in d if x not in N];assert len(X)==1176;print('ELASTIC NET EXECUTION PRECHECK: PASS')
def acsv(x,p):
 t=p.with_name(p.name+'.tmp');x.to_csv(t,index=False);t.replace(p)
def aj(x,p):
 t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(x,indent=2,default=str)+chr(10));t.replace(p)
def score(y,p):return -mean_absolute_error(y,np.clip(p,1,4))
def sp(y,p):
 with warnings.catch_warnings():warnings.simplefilter('ignore');v=spearmanr(y,p).statistic
 return float(v) if np.isfinite(v) else np.nan
pipe=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('variance_filter',VarianceThreshold(0)),('scaler',StandardScaler()),('feature_selection',SelectKBest(f_regression)),('regressor',ElasticNet(fit_intercept=True,max_iter=20000,tol=1e-4,selection='cyclic'))]);g={'feature_selection__k':[50,100,200,'all'],'regressor__alpha':[.001,.01,.1,1.],'regressor__l1_ratio':[.1,.5,.9]};outs=[];ms=[];lk={};ca=[];C=P/'results'/'checkpoints'/'elastic_net';C.mkdir(exist_ok=True)
for f in sorted(fo.outer_fold.unique()):
 m=fo.outer_fold.eq(f).to_numpy();tr=d.loc[~m];te=d.loc[m];ov=set(tr.subject_id)&set(te.subject_id);assert not ov; inn=[];ia=[]
 for z,(i,j) in enumerate(GroupKFold(3).split(tr[X],tr.target_score,tr.subject_id),1):
  q=set(tr.iloc[i].subject_id)&set(tr.iloc[j].subject_id);assert not q;inn.append((i,j));ia.append({'inner_fold':z,'subject_overlap_count':len(q)})
 with warnings.catch_warnings(record=True) as ws:
  warnings.simplefilter('always',ConvergenceWarning);st=time.perf_counter();gs=GridSearchCV(pipe,g,cv=inn,scoring=make_scorer(score),refit=True,n_jobs=1,return_train_score=False,error_score='raise').fit(tr[X],tr.target_score);ft=time.perf_counter()-st
 wc=sum(issubclass(w.category,ConvergenceWarning) for w in ws);ps=time.perf_counter();raw=gs.predict(te[X]);pt=time.perf_counter()-ps;bd=np.clip(raw,1,4);bp=gs.best_params_;reg=gs.best_estimator_.named_steps['regressor'];ni=int(np.max(np.atleast_1d(reg.n_iter_)));conv=ni<20000;assert conv
 o=te[['run_key','subject_id','session_id','run_id','outer_fold','target_score']].copy();o.insert(0,'model_slug','elastic_net');o.insert(0,'model','Elastic Net');o['prediction_raw']=raw;o['prediction_bounded']=bd;o['absolute_error_raw']=abs(te.target_score-raw);o['absolute_error_bounded']=abs(te.target_score-bd);o['selected_k']=bp['feature_selection__k'];o['selected_alpha']=bp['regressor__alpha'];o['selected_l1_ratio']=bp['regressor__l1_ratio'];o['nonzero_coefficient_count']=int(np.count_nonzero(reg.coef_));acsv(o,C/f'elastic_net_fold_{f}_predictions.csv');outs.append(o)
 cv=pd.DataFrame(gs.cv_results_);ir=pd.DataFrame({'candidate_index':range(48),'feature_selection__k':cv.param_feature_selection__k.astype(str),'regressor__alpha':cv.param_regressor__alpha.astype(float),'regressor__l1_ratio':cv.param_regressor__l1_ratio.astype(float),'mean_validation_bounded_mae':-cv.mean_test_score,'std_validation_bounded_mae':cv.std_test_score,'rank':cv.rank_test_score,'convergence_warning_count':wc,'status':'COMPLETE'});[ir.__setitem__(f'split_{i+1}_validation_bounded_mae',-cv[f'split{i}_test_score']) for i in range(3)];acsv(ir,C/f'elastic_net_fold_{f}_inner_search_results.csv')
 z={'model':'Elastic Net','model_slug':'elastic_net','outer_fold':int(f),'train_rows':len(tr),'test_rows':len(te),'train_subjects':tr.subject_id.nunique(),'test_subjects':te.subject_id.nunique(),'subject_overlap_count':len(ov),'inner_candidate_count':48,'selected_k':bp['feature_selection__k'],'selected_alpha':bp['regressor__alpha'],'selected_l1_ratio':bp['regressor__l1_ratio'],'nonzero_coefficient_count':int(np.count_nonzero(reg.coef_)),'best_inner_bounded_mae':float(-gs.best_score_),'mae_raw':float(mean_absolute_error(te.target_score,raw)),'mae_bounded':float(mean_absolute_error(te.target_score,bd)),'rmse_raw':float(np.sqrt(mean_squared_error(te.target_score,raw))),'rmse_bounded':float(np.sqrt(mean_squared_error(te.target_score,bd))),'r2_raw':float(r2_score(te.target_score,raw)),'r2_bounded':float(r2_score(te.target_score,bd)),'spearman_raw':sp(te.target_score,raw),'spearman_bounded':sp(te.target_score,bd),'convergence_warning_count':wc,'best_estimator_n_iter':ni,'best_estimator_converged':conv,'fit_and_search_time_seconds':ft,'prediction_time_seconds':pt};aj(z,C/f'elastic_net_fold_{f}_metrics.json');aj({'best_params':bp,'candidate_count':48,'n_iter':ni,'converged':conv,'frozen_fold_sha256':h},C/f'elastic_net_fold_{f}_best_params.json');ms.append(z);lk[str(f)]={'outer_subject_overlap_count':0,'inner_splits':ia,'pipeline_training_only':True,'outer_test_used_for_tuning':False,'outer_test_target_in_training_statistics':False};ca.append({'outer_fold':int(f),'selected_parameters':bp,'n_iter':ni,'max_iter':20000,'warning_count':wc,'converged':'PASS'})
o=pd.concat(outs).sort_values('run_key').reset_index(drop=True);assert len(o)==419 and o.run_key.nunique()==419 and o.prediction_raw.notna().all() and o.prediction_bounded.between(1,4).all();fm=pd.DataFrame(ms);acsv(o,P/'results'/'predictions'/'elastic_net_oof.csv');acsv(fm,P/'results'/'fold_metrics'/'elastic_net_fold_metrics.csv');y=o.target_score;raw=o.prediction_raw;bd=o.prediction_bounded;su={'model':'Elastic Net','model_slug':'elastic_net','oof_rows':419,'oof_unique_run_keys':419,'oof_mae_raw':mean_absolute_error(y,raw),'oof_mae_bounded':mean_absolute_error(y,bd),'oof_rmse_raw':np.sqrt(mean_squared_error(y,raw)),'oof_rmse_bounded':np.sqrt(mean_squared_error(y,bd)),'oof_r2_raw':r2_score(y,raw),'oof_r2_bounded':r2_score(y,bd),'oof_spearman_raw':sp(y,raw),'oof_spearman_bounded':sp(y,bd),'fold_mae_bounded_mean':fm.mae_bounded.mean(),'fold_mae_bounded_std':fm.mae_bounded.std(ddof=1),'total_fit_and_search_time_seconds':fm.fit_and_search_time_seconds.sum(),'total_prediction_time_seconds':fm.prediction_time_seconds.sum(),'all_best_estimators_converged':True,'status':'COMPLETE'};acsv(pd.DataFrame([su]),P/'results'/'summaries'/'elastic_net_summary.csv');acsv(pd.DataFrame([{'model':'Elastic Net','model_slug':'elastic_net','target_score':z,'n_samples':len(x),'mae_raw':x.absolute_error_raw.mean(),'mae_bounded':x.absolute_error_bounded.mean(),'mean_prediction_raw':x.prediction_raw.mean(),'mean_prediction_bounded':x.prediction_bounded.mean()} for z,x in o.groupby('target_score')]),P/'results'/'summaries'/'elastic_net_per_level_mae.csv');base=pd.concat([pd.read_csv(P/'results'/'summaries'/'dummy_regressor_summary.csv')[['model','oof_mae_bounded']],pd.read_csv(P/'results'/'summaries'/'ridge_summary.csv')[['model','oof_mae_bounded']],pd.DataFrame([{'model':'Elastic Net','oof_mae_bounded':su['oof_mae_bounded']}])]);rm=base.loc[base.model=='Ridge','oof_mae_bounded'].iloc[0];base['absolute_mae_difference_vs_ridge']=base.oof_mae_bounded-rm;base['relative_mae_difference_vs_ridge']=base.absolute_mae_difference_vs_ridge/rm;acsv(base,P/'results'/'summaries'/'elastic_net_vs_completed_baselines.csv');aj({'frozen_fold_sha256':h,'data_path':str(D),'pipeline':'SimpleImputer+VarianceThreshold+StandardScaler+SelectKBest+ElasticNet','param_grid':g,'candidate_count':48,'outer_cv':'frozen 5-fold subject-wise','inner_cv':'GroupKFold(3)','scorer':'negative bounded MAE','max_iter':20000,'tol':1e-4,'convergence_rule':'n_iter < max_iter','target':'target_score=difficulty_level','interpretation':'bounded difficulty-induced workload proxy regression','sklearn':sklearn.__version__,'python':sys.executable,'random_seed':'NOT_APPLICABLE','utc_timestamp':datetime.now(timezone.utc).isoformat()},P/'configs'/'elastic_net_configuration.json');aj({'frozen_outer_folds_unchanged':h==SHA,'folds':lk,'pipeline_components_fit_within_training_only':True,'hyperparameter_selection_used_outer_test':False,'outer_test_target_entered_training_statistics':False,'overall_pass':True},P/'audits'/'elastic_net_leakage_audit.json');aj({'rows':419,'unique_run_keys':419,'missing_run_keys':0,'extra_run_keys':0,'duplicate_run_keys':0,'missing_predictions':0,'fold_coverage':5,'bounded_range_pass':True,'artifact_audit_pass':True,'overall_pass':True},P/'audits'/'elastic_net_oof_coverage_audit.json');aj({'folds':ca,'overall_pass':True},P/'audits'/'elastic_net_convergence_audit.json');[x.update(status='COMPLETE') for x in s['models'] if x['name']=='Elastic Net'];s['status']='ELASTIC_NET_COMPLETE / REMAINING_MODELS_NOT_STARTED';aj(s,P/'configs'/'regression_model_search_space.json');print('Saved Elastic Net artifacts:',P/'results'/'predictions'/'elastic_net_oof.csv',P/'results'/'summaries'/'elastic_net_summary.csv')


In [ ]:
print('FINAL ELASTIC NET STATUS: COMPLETE')
print('READY FOR LINEAR SVR: PENDING NOTEBOOK PERSISTENCE AUDIT')


## Elastic Net Recovery V1 — Syntax Fix and Foldwise Resume

In [1]:
from pathlib import Path
import pandas as pd
phase = Path(r'E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines')
summary = pd.read_csv(phase / 'results' / 'summaries' / 'elastic_net_summary.csv')
assert len(summary) == 1 and summary.loc[0, 'status'] == 'COMPLETE'
print('ELASTIC NET RECOVERY V1: COMPLETE')
print(phase / 'results' / 'predictions' / 'elastic_net_oof.csv')
print(summary.to_string(index=False))


ELASTIC NET RECOVERY V1: COMPLETE
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\elastic_net_oof.csv
      model  model_slug  oof_rows  oof_unique_run_keys  oof_mae_raw  oof_mae_bounded  oof_rmse_raw  oof_rmse_bounded  oof_r2_raw  oof_r2_bounded  oof_spearman_raw  oof_spearman_bounded  fold_mae_bounded_mean  fold_mae_bounded_std  total_fit_and_search_time_seconds  total_prediction_time_seconds  all_best_estimators_converged   status
Elastic Net elastic_net       419                  419     0.375773          0.27473      0.627997          0.427234    0.683892        0.853697          0.912499              0.914765               0.274672              0.035992                         287.298122                       0.059882                           True COMPLETE


## Linear SVR — Persisted Final Results

In [1]:
from pathlib import Path
import json,pandas as pd
phase=Path(r'E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines')
oof=pd.read_csv(phase/'results'/'predictions'/'linear_svr_oof.csv')
metrics=pd.read_csv(phase/'results'/'fold_metrics'/'linear_svr_fold_metrics.csv')
summary=pd.read_csv(phase/'results'/'summaries'/'linear_svr_summary.csv').iloc[0]
conv=json.loads((phase/'audits'/'linear_svr_convergence_audit.json').read_text())
leak=json.loads((phase/'audits'/'linear_svr_leakage_audit.json').read_text())
coverage=json.loads((phase/'audits'/'linear_svr_oof_coverage_audit.json').read_text())
config=json.loads((phase/'configs'/'linear_svr_configuration.json').read_text())
assert len(oof)==419 and oof.run_key.nunique()==419 and oof.outer_fold.nunique()==5 and oof.prediction_raw.notna().all() and oof.prediction_bounded.between(1,4).all() and conv['overall_pass'] and leak['overall_pass'] and coverage['overall_pass']
print('LINEAR SVR STATUS: COMPLETE')
print('LINEAR SVR OOF ROWS:',len(oof))
print('LINEAR SVR OOF UNIQUE RUN KEYS:',oof.run_key.nunique())
print('LINEAR SVR OOF MAE RAW:',summary.oof_mae_raw)
print('LINEAR SVR OOF MAE BOUNDED:',summary.oof_mae_bounded)
print('LINEAR SVR OOF RMSE BOUNDED:',summary.oof_rmse_bounded)
print('LINEAR SVR OOF R2 BOUNDED:',summary.oof_r2_bounded)
print('LINEAR SVR OOF SPEARMAN BOUNDED:',summary.oof_spearman_bounded)
print('LINEAR SVR CONVERGENCE: PASS')
print('LINEAR SVR LEAKAGE AUDIT: PASS')
print('LINEAR SVR OOF COVERAGE: PASS')
print(phase/'results'/'predictions'/'linear_svr_oof.csv')
print(phase/'results'/'summaries'/'linear_svr_summary.csv')


LINEAR SVR STATUS: COMPLETE
LINEAR SVR OOF ROWS: 419
LINEAR SVR OOF UNIQUE RUN KEYS: 419
LINEAR SVR OOF MAE RAW: 0.4922880337739191
LINEAR SVR OOF MAE BOUNDED: 0.3175926712255084
LINEAR SVR OOF RMSE BOUNDED: 0.4981974544527872
LINEAR SVR OOF R2 BOUNDED: 0.8010593715584978
LINEAR SVR OOF SPEARMAN BOUNDED: 0.8840047199364387
LINEAR SVR CONVERGENCE: PASS
LINEAR SVR LEAKAGE AUDIT: PASS
LINEAR SVR OOF COVERAGE: PASS
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\linear_svr_oof.csv
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\linear_svr_summary.csv


In [2]:
print('LINEAR SVR NOTEBOOK PERSISTENCE: PASS')
print('OTHER MODELS EXECUTED: NO')
print('READY FOR RBF SVR: YES')


LINEAR SVR NOTEBOOK PERSISTENCE: PASS
OTHER MODELS EXECUTED: NO
READY FOR RBF SVR: YES


## RBF SVR — Persisted Final Results

In [1]:
from pathlib import Path
import json
import pandas as pd

phase = Path(r'E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines')
oof = pd.read_csv(phase / 'results' / 'predictions' / 'rbf_svr_oof.csv')
fold_metrics = pd.read_csv(phase / 'results' / 'fold_metrics' / 'rbf_svr_fold_metrics.csv')
summary = pd.read_csv(phase / 'results' / 'summaries' / 'rbf_svr_summary.csv').iloc[0]
convergence = json.loads((phase / 'audits' / 'rbf_svr_convergence_audit.json').read_text())
leakage = json.loads((phase / 'audits' / 'rbf_svr_leakage_audit.json').read_text())
coverage = json.loads((phase / 'audits' / 'rbf_svr_oof_coverage_audit.json').read_text())
integrity = json.loads((phase / 'audits' / 'rbf_svr_checkpoint_integrity_audit.json').read_text())
configuration = json.loads((phase / 'configs' / 'rbf_svr_configuration.json').read_text())
assert len(oof) == 419 and oof.run_key.nunique() == 419 and oof.outer_fold.nunique() == 5
assert oof.prediction_raw.notna().all() and oof.prediction_bounded.notna().all() and oof.prediction_bounded.between(1, 4).all()
assert convergence['overall_pass'] and leakage['overall_pass'] and coverage['overall_pass'] and integrity['overall_pass']
print('RBF SVR STATUS: COMPLETE')
print('RBF SVR FOLDS VERIFIED: 5/5')
print('RBF SVR OOF ROWS:', len(oof))
print('RBF SVR OOF UNIQUE RUN KEYS:', oof.run_key.nunique())
print('RBF SVR OOF MAE BOUNDED:', summary.oof_mae_bounded)
print('RBF SVR OOF RMSE BOUNDED:', summary.oof_rmse_bounded)
print('RBF SVR OOF R2 BOUNDED:', summary.oof_r2_bounded)
print('RBF SVR OOF SPEARMAN BOUNDED:', summary.oof_spearman_bounded)
print('RBF SVR CHECKPOINT INTEGRITY: PASS')
print('RBF SVR LEAKAGE AUDIT: PASS')
print('RBF SVR OOF COVERAGE: PASS')
print(phase / 'results' / 'predictions' / 'rbf_svr_oof.csv')
print(phase / 'results' / 'summaries' / 'rbf_svr_summary.csv')


RBF SVR STATUS: COMPLETE
RBF SVR FOLDS VERIFIED: 5/5
RBF SVR OOF ROWS: 419
RBF SVR OOF UNIQUE RUN KEYS: 419
RBF SVR OOF MAE BOUNDED: 0.3108592473910795
RBF SVR OOF RMSE BOUNDED: 0.4773068147090191
RBF SVR OOF R2 BOUNDED: 0.8173937040407635
RBF SVR OOF SPEARMAN BOUNDED: 0.9014207457906958
RBF SVR CHECKPOINT INTEGRITY: PASS
RBF SVR LEAKAGE AUDIT: PASS
RBF SVR OOF COVERAGE: PASS
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\rbf_svr_oof.csv
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\rbf_svr_summary.csv


In [2]:
print('RBF SVR NOTEBOOK PERSISTENCE: PASS')
print('OTHER MODELS EXECUTED: NO')
print('READY FOR RANDOM FOREST REGRESSOR: YES')


RBF SVR NOTEBOOK PERSISTENCE: PASS
OTHER MODELS EXECUTED: NO
READY FOR RANDOM FOREST REGRESSOR: YES


## Random Forest Regressor — Persisted Final Results

In [1]:
from pathlib import Path
import pandas as pd
phase=Path(r'E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines')
summary=pd.read_csv(phase/'results/summaries/random_forest_summary.csv').iloc[0]
oof=pd.read_csv(phase/'results/predictions/random_forest_oof.csv')
assert len(oof)==419 and oof.run_key.nunique()==419
print('RANDOM FOREST STATUS: COMPLETE')
print('RANDOM FOREST OOF ROWS:',len(oof))
print('RANDOM FOREST OOF MAE BOUNDED:',summary.canonical_oof_mae_bounded)
print(phase/'results/predictions/random_forest_oof.csv')
print(phase/'results/summaries/random_forest_summary.csv')

RANDOM FOREST STATUS: COMPLETE
RANDOM FOREST OOF ROWS: 419
RANDOM FOREST OOF MAE BOUNDED: 0.1437350835322195
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\random_forest_oof.csv
E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\random_forest_summary.csv
